In [1]:
"""
Reference code to reproduce the two tables (Table 1 / Table 2).

Assumptions about inputs:
- total_scenarios.csv: one row per (scenario, run), containing metrics for a given source set / budget.
  Expected columns (same as in our pipeline outputs):
    scenario, name, run_mode, source_set,
    n_expected_steps,
    step_recall, chain_recall,
    step_precision, chain_precision,
    reconstructability,
    n_chain_steps, n_tagged_steps,
    n_events (only needed for Table 2 totals)
- For Table 2 attack/benign counts: need per-event normalized telemetry files
  (one file per scenario) with an attack label column such as is_attack or label.

Key principles:
- Coverage/Recall metrics are *weighted by expected steps* (wtd.):
      wtd_metric = sum_s( metric_s * n_expected_steps_s ) / sum_s( n_expected_steps_s )
- For "avg over ..." categories: we first average within each scenario (so a scenario doesn't
  get more weight just because it has more runs/pairs), then we aggregate across scenarios.
- For precision/reconstruction marked "(mean)": we compute scenario-level means and then take
  a simple mean across scenarios. NaN precision (undefined when there are no predictions) is
  treated as 0, which matches typical paper-table reporting for these cases.
"""

from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence

import numpy as np
import pandas as pd


# -----------------------------
# Utility helpers
# -----------------------------
def mean_nan_as_zero(series: pd.Series) -> float:
    """Compute mean with NaN treated as 0 (useful for precision/recon 'mean' columns)."""
    return float(pd.to_numeric(series, errors="coerce").fillna(0.0).mean())


def scenario_mean(df: pd.DataFrame, col: str) -> pd.Series:
    """
    Scenario-level mean for a column.
    We group by scenario and average runs within each scenario.
    """
    return df.groupby("scenario")[col].apply(mean_nan_as_zero)


def scenario_first(df: pd.DataFrame, col: str) -> pd.Series:
    """Pick the first value per scenario (for constants like n_expected_steps)."""
    return df.groupby("scenario")[col].first()


def weighted_by_expected_steps(values_per_scenario: pd.Series, n_expected_steps: pd.Series) -> float:
    """
    Weighted mean by expected steps (paper's 'wtd.' definition):
        sum_s( value_s * w_s ) / sum_s( w_s ), where w_s = n_expected_steps_s
    """
    v = pd.to_numeric(values_per_scenario, errors="coerce").fillna(0.0)
    w = pd.to_numeric(n_expected_steps, errors="coerce").fillna(0.0)
    denom = float(w.sum())
    return float((v * w).sum() / denom) if denom > 0 else 0.0


def canon_source_set(s: str) -> str:
    """
    Canonicalize a source_set string like 'zeek+auditd' to 'auditd+zeek'
    so matching is consistent.
    """
    if not isinstance(s, str):
        return ""
    parts = [p.strip() for p in s.split("+") if p.strip()]
    return "+".join(sorted(parts))


def sources_of(s: str) -> set:
    """Return a set of sources in a canonical source_set string."""
    return set([p for p in canon_source_set(s).split("+") if p])


def pick_best_per_scenario(df: pd.DataFrame) -> pd.DataFrame:
    """
    Pick the "best" row per scenario (used for 'best single-source' etc.)
    Tie-breaking matches the typical logic we used before:
    1) higher reconstructability is better
    2) tie-break: higher n_chain_steps is better
    3) tie-break: higher n_tagged_steps is better
    """
    sort_cols = ["reconstructability", "n_chain_steps", "n_tagged_steps"]
    tmp = df.copy()
    for c in sort_cols:
        if c not in tmp.columns:
            tmp[c] = 0
    tmp = tmp.sort_values(sort_cols, ascending=[False, False, False])
    return tmp.groupby("scenario", as_index=False).head(1)


@dataclass
class AggResult:
    """Aggregated metrics in the exact shape needed for Table 1."""
    n_scenarios: int
    tag_cov_wtd: float
    chain_cov_wtd: float
    stepR_wtd: float
    chainR_wtd: float
    stepP_mean: float
    chainP_mean: float
    recon_mean: float


def aggregate_like_table1(df: pd.DataFrame, *, scenario_inner_mean: bool) -> AggResult:
    """
    Core aggregation function to reproduce Table 1.

    - If scenario_inner_mean=True:
        First compute scenario-level mean across runs, then aggregate across scenarios.
        This is the correct interpretation for rows like:
            'avg over all single sources'
            'avg over 2-source pair'
        (prevents scenarios with more runs/pairs from dominating the global mean).

    - If scenario_inner_mean=False:
        df is expected to already be one row per scenario (e.g., after best-per-scenario selection).
        We still group-by scenario once for safety.
    """
    if len(df) == 0:
        return AggResult(0, 0, 0, 0, 0, 0, 0, 0)

    # Ensure required columns exist (missing -> NaN)
    needed = [
        "scenario", "n_expected_steps",
        "step_recall", "chain_recall",
        "step_precision", "chain_precision",
        "reconstructability",
    ]
    df = df.copy()
    for c in needed:
        if c not in df.columns:
            df[c] = np.nan

    df["scenario"] = df["scenario"].astype(str)

    # Scenario-level series
    stepR_s = scenario_mean(df, "step_recall")
    chainR_s = scenario_mean(df, "chain_recall")
    stepP_s = scenario_mean(df, "step_precision")
    chainP_s = scenario_mean(df, "chain_precision")
    recon_s = scenario_mean(df, "reconstructability")
    n_steps_s = scenario_first(df, "n_expected_steps")

    n_sc = int(stepR_s.index.nunique())

    # Paper uses Tag Coverage and Step Recall with the same wtd. definition,
    # and Chain Coverage and Chain Recall similarly.
    stepR_wtd = weighted_by_expected_steps(stepR_s, n_steps_s)
    chainR_wtd = weighted_by_expected_steps(chainR_s, n_steps_s)

    return AggResult(
        n_scenarios=n_sc,
        tag_cov_wtd=stepR_wtd,
        chain_cov_wtd=chainR_wtd,
        stepR_wtd=stepR_wtd,
        chainR_wtd=chainR_wtd,
        stepP_mean=float(stepP_s.mean()) if len(stepP_s) else 0.0,
        chainP_mean=float(chainP_s.mean()) if len(chainP_s) else 0.0,
        recon_mean=float(recon_s.mean()) if len(recon_s) else 0.0,
    )


def latex_highlight_best_second(df: pd.DataFrame, metric_cols: Sequence[str]) -> pd.DataFrame:
    """
    Optional: convert best/second-best values per metric column to LaTeX markup:
      best   -> \\textbf{...}
      second -> \\underline{...}
    """
    out = df.copy()
    for c in metric_cols:
        vals = pd.to_numeric(df[c], errors="coerce").fillna(-np.inf).values
        uniq = sorted({v for v in vals if np.isfinite(v)}, reverse=True)
        if not uniq:
            continue
        best = uniq[0]
        second = uniq[1] if len(uniq) > 1 else None

        def fmt(v: float) -> str:
            if not np.isfinite(v):
                return "0.000"
            s = f"{v:.3f}"
            if abs(v - best) < 1e-12:
                return f"\\textbf{{{s}}}"
            if second is not None and abs(v - second) < 1e-12:
                return f"\\underline{{{s}}}"
            return s

        out[c] = [fmt(v) for v in vals]
    return out



In [2]:
import re

COMPOSITE_SOURCE_WEIGHTS = {"azure_events": 3}

SCENARIO_ID_COLS = ["scenario", "scenario_id", "scn", "sc"]
SCENARIO_NAME_COLS = ["scenario_name", "scenario_title", "name"]
MODE_COLS = ["mode", "budget_mode", "run_mode"]
SOURCE_SET_COLS = ["source_set", "sources", "source", "source_combo"]
TOTAL_COLS = [
    "n_ingested_events", "n_events_ingested",
    "total_records", "n_total_records",
    "n_records", "n_events", "records", "total"
]
HITS_COLS = ["tagger_total_hits", "total_hits", "n_hits", "tagger_hits"]


def pick_existing_col(df: pd.DataFrame, candidates: list[str], required: bool = True) -> str | None:
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"None of these columns exist: {candidates}. Available={list(df.columns)}")
    return None


def effective_source_count(source_set: str) -> int:
    parts = [p.strip() for p in str(source_set).split("+") if p.strip()]
    n = len(parts)
    for p in parts:
        if p in COMPOSITE_SOURCE_WEIGHTS:
            n += (COMPOSITE_SOURCE_WEIGHTS[p] - 1)
    return n


def scenario_index(s: str) -> str:
    m = re.search(r"(\d+)", str(s))
    return m.group(1) if m else str(s)


def choose_row_for_stats(
    g: pd.DataFrame,
    total_col: str,
    hits_col: str,
    mode_col: str | None,
    source_col: str | None,
) -> pd.Series:
    """
    Selection rule that matches your Table:
    - primary: max Total Records
    - secondary: max tagger_total_hits  (so Attack Records aligns)
    - tie: prefer mode=='multi'
    - tie: prefer larger effective source count
    """
    gg = g.copy()
    gg[total_col] = pd.to_numeric(gg[total_col], errors="coerce").fillna(0.0)
    gg[hits_col] = pd.to_numeric(gg[hits_col], errors="coerce").fillna(0.0)

    # 1) max total
    max_total = gg[total_col].max()
    gg = gg[gg[total_col] == max_total].copy()

    # 2) max hits among tied-total rows
    max_hits = gg[hits_col].max()
    gg = gg[gg[hits_col] == max_hits].copy()

    # 3) prefer multi
    if mode_col and mode_col in gg.columns:
        gg["_is_multi"] = (gg[mode_col].astype(str).str.lower() == "multi").astype(int)
    else:
        gg["_is_multi"] = 0

    # 4) prefer larger effective source count
    if source_col and source_col in gg.columns:
        gg["_n_eff_sources"] = gg[source_col].apply(effective_source_count)
    else:
        gg["_n_eff_sources"] = 0

    gg = gg.sort_values(["_is_multi", "_n_eff_sources"], ascending=[False, False])
    return gg.iloc[0]


def build_stats_table(csv_path: str | Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    scn_id_col = pick_existing_col(df, SCENARIO_ID_COLS, required=True)
    scn_name_col = pick_existing_col(df, SCENARIO_NAME_COLS, required=False)
    mode_col = pick_existing_col(df, MODE_COLS, required=False)
    source_col = pick_existing_col(df, SOURCE_SET_COLS, required=False)
    total_col = pick_existing_col(df, TOTAL_COLS, required=True)
    hits_col = pick_existing_col(df, HITS_COLS, required=True)

    df[total_col] = pd.to_numeric(df[total_col], errors="coerce").fillna(0.0)
    df[hits_col] = pd.to_numeric(df[hits_col], errors="coerce").fillna(0.0)

    rows = []
    for sid, g in df.groupby(scn_id_col, sort=True):
        r = choose_row_for_stats(g, total_col, hits_col, mode_col, source_col)

        total = float(r[total_col])
        hits = float(r[hits_col])
        hits = min(hits, total)  # clamp
        benign = max(total - hits, 0.0)
        pct = (hits / total * 100.0) if total > 0 else 0.0

        scen_name = (r[scn_name_col] if scn_name_col else str(sid))
        rows.append({
            "Scenario": f"{scenario_index(sid)}.{scen_name}",
            "Total Records": int(round(total)),
            "Benign Records": int(round(benign)),
            "Attack Records": int(round(hits)),
            "Attack %": pct,
            "Chosen mode": (r[mode_col] if mode_col else None),
            "Chosen source_set": (r[source_col] if source_col else None),
        })

    return pd.DataFrame(rows)


def to_latex(stats: pd.DataFrame, label="tab:collected_data_stats") -> str:
    lines = []
    for _, r in stats.iterrows():
        lines.append(
            f"{r['Scenario']} & {r['Total Records']:,} & {r['Benign Records']:,} & "
            f"{r['Attack Records']:,} ({r['Attack %']:.2f}\\%) \\\\"
        )
    body = "\n".join(lines)

    return rf"""\begin{{table}}[t]
\centering
\small
\caption{{Statistics of Collected Telemetry (Normalized)}}
\label{{{label}}}
\begin{{tabular}}{{@{{}}lrrr@{{}}}}
\toprule
\textbf{{Scenario}} & \textbf{{Total Records}} & \textbf{{Benign Records}} & \textbf{{Attack Records}} \\
\midrule
{body}
\bottomrule
\end{{tabular}}
\end{{table}}
"""




In [3]:
import pandas as pd

pd.set_option("display.max_rows", None)        
pd.set_option("display.max_columns", None)    
pd.set_option("display.width", None)           
pd.set_option("display.max_colwidth", None)    


In [6]:

# ===== Table 2 =====
import sys
csv_path = "total_scenarios_old.csv"
stats = build_stats_table(csv_path)

print(stats[["Scenario", "Total Records", "Benign Records", "Attack Records", "Attack %", "Chosen mode", "Chosen source_set"]])
print("\n=== LaTeX ===")
print(to_latex(stats))


     Scenario  Total Records  Benign Records  Attack Records   Attack %  \
0   1.Stegano           8534            8530               4   0.046871   
1   2.Starter          53978           48788            5190   9.615028   
2  3.Parallel          88674           41270           47404  53.458736   
3     4.NPMEX         188270          103925           84345  44.800021   
4       5.3CX           7453            6881             572   7.674762   
5   6.CloudEX           9774            9636             138   1.411909   
6  7.LayerInj         222046          169894           52152  23.487025   

  Chosen mode                                                Chosen source_set  
0       multi  azure_conn+azure_process+azure_security+azure_events+azure_port  
1       multi                                              azure_events+syslog  
2       multi                                 auditd+auth+suricata+syslog+zeek  
3       multi                                 auditd+auth+suricata+syslog+z